# Baseline Example

In [ ]:
from IPython.display import display, Audio

display(Audio(audio, rate=sr))
display(Audio(true_audio, rate=sr))
display(Audio(denoised, rate=sr))

# Example Data Augmentation Samples

In [ ]:
class RandomNoiseTransform:
    def __init__(self, sample_rate, noise_dir, min_snr_db=0, max_snr_db=15):
        self.sample_rate = sample_rate
        self.min_snr_db = min_snr_db
        self.max_snr_db = max_snr_db
        
        self.noise_files_list = list(glob.glob(f'{noise_dir}/**/*.wav'))

    def __call__(self, audio_data):
        random_noise_file = random.choice(self.noise_files_list)
        effects = [
            ['remix', '1'], # convert to mono
            ['rate', str(self.sample_rate)], # resample
        ]
        noise, _ = torchaudio.sox_effects.apply_effects_file(random_noise_file, effects, normalize=True)
        audio_length = audio_data.shape[-1]
        noise_length = noise.shape[-1]
        if noise_length > audio_length:
            offset = random.randint(0, noise_length-audio_length)
            noise = noise[..., offset:offset+audio_length]
        elif noise_length < audio_length:
            noise = torch.cat([noise, torch.zeros((noise.shape[0], audio_length-noise_length))], dim=-1)

        snr_db = random.randint(self.min_snr_db, self.max_snr_db)
        snr = math.exp(snr_db / 10)
        audio_power = audio_data.norm(p=2)
        noise_power = noise.norm(p=2)
        scale = snr * noise_power / audio_power

        return (scale * audio_data + noise ) / 2


random_noise_transform = RandomNoiseTransform(sample_rate=8000, noise_dir="/kaggle/input/musan-noise/musan/noise")

transformed_audio = random_noise_transform(torch.as_tensor(noisy).unsqueeze(0))

display(HTML("<span>Clean</span>"))
display(Audio(noisy, rate=8000))

display(HTML("<span>Noisy 1</span>"))
display(Audio(transformed_audio[0], rate=8000))

transformed_audio = random_noise_transform(torch.as_tensor(noisy).unsqueeze(0))
display(HTML("<span>Noisy 2</span>"))
display(Audio(transformed_audio[0], rate=8000))

transformed_audio = random_noise_transform(torch.as_tensor(noisy).unsqueeze(0))
display(HTML("<span>Noisy 3</span>"))
display(Audio(transformed_audio[0], rate=8000))

# Example predictions for augmented model

In [ ]:
model = Unet1D(in_channels=1, out_channels=1)
trainer = Trainer(experiment=f"augmented_l2_{model.name}", model=model, loss_fn=torch.nn.functional.mse_loss, batch_size=8, num_workers=0)
trainer.fit(train_dataset=train_dataset_augmented, val_dataset=val_dataset_augmented,epochs=2)

# load best weights and set it to eval mode
model.load_state_dict(torch.load(trainer.model_checkpoint_path))
model.eval()

sample = test_dataset[0]
denoised = unet1d_denoiser(model)(sample["noisy"][0], sr=test_dataset.sample_rate)
display_prediction(denoised, sample["noisy"], sample["clean"], sr=test_dataset.sample_rate)

metrics = evaluate(DATA_ROOT + "/test", unet1d_denoiser(model)) # dummy evaluation
metrics.to_csv(trainer.experiment_directory + "/metrics.csv", index=False)
metrics.describe()

Validation: 100%|██████████| 53/53 [00:16<00:00,  3.26it/s]


Epoch 1/2 - Train Loss: 0.0005, Val Loss: 0.0004


Validation: 100%|██████████| 53/53 [00:16<00:00,  3.27it/s]
/tmp/ipykernel_23/2787670255.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load

Epoch 2/2 - Train Loss: 0.0004, Val Loss: 0.0004


100%|██████████| 782/782 [02:34<00:00,  5.07it/s]


,sdr,nsdr,si_snr,pesq
count,782.000000,782.000000,782.000000,782.000000
mean,15.705292,5.760000,14.856874,2.619593
std,2.800071,3.877642,2.758832,0.543223
min,7.350418,-6.920510,3.925513,1.375478
25%,13.570387,2.890464,12.944853,2.176471
50%,16.413998,6.084603,15.479604,2.634216
75%,17.779359,8.828224,16.884389,3.059165
max,20.945803,13.459986,19.765642,3.785928
